# 🖥️ Which RunPod GPU should I use?

This notebook loads **bottlecapai/ThinkingCap-Qwen3.6-27B** (~27 B params, a *reasoning* / "thinking" model). The VRAM you need depends on the **precision** you pick in the selector below — choose a pod that clears the *VRAM* column:

| Precision | Weights | ~VRAM (with overhead) | Recommended RunPod GPU | ~Price / hr |
|:---|:---|:---|:---|:---|
| **4-bit** (NF4) | ~14 GB | ~18–20 GB | **RTX A5000 · 24 GB** (or RTX 4090 24 GB) | **\$0.27** |
| **16-bit** (bf16) | ~54 GB | ~58–62 GB | **A100 SXM · 80 GB** (or H100 80 GB) | **\$1.19** |
| **32-bit** (fp32) | ~108 GB | ~112–118 GB | **2× A100 · 80 GB** (`device_map="auto"`) | **~\$2.38** |

_Prices: RunPod Community Cloud, on-demand, ~July 2026 — they change often, check [runpod.io/pricing](https://www.runpod.io/pricing). Also set the pod's **Container/Volume disk to ≥ 150 GB** (27 B fp32 shards are large)._

> **Note:** `ThinkingCap-Qwen3.6-27B` is a full-precision, instruction-tuned **reasoning** model built on Qwen3.6 — it emits an internal chain-of-thought (often wrapped in `<think>…</think>`) *before* its final answer. That means responses run longer, so the test cells below use a generous `max_new_tokens`. The **27B** is the parameter count, **not** 4-bit; run the 4-bit precision cell below if you want quantized weights.

# Install libraries

In [2]:
%pip install torch
%pip install --upgrade transformers
%pip install bitsandbytes
%pip install accelerate
%pip install peft
%pip install datasets
%pip install evaluate
%pip install trl
%pip install matplotlib
%pip install tensorboard
%pip install sentencepiece

Note: you may need to restart the kernel to use updated packages.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/11.5 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 119.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 22.8 MB/s  0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/4.5 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 139.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/3.3 MB ? eta -:--:--

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━ 1.3/1.4 MB 59.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━ 1.3/1.4 MB 59.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━ 1.3/1.4 MB 59.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 904.8 kB/s  0:00:01


Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_properties(i).name)

NVIDIA A100 80GB PCIe


In [3]:
from huggingface_hub import login

API_TOKEN = "HF_TOKEN_REMOVED"
login(token=API_TOKEN)

# ⚙️ Choose precision -- run ONE of the three cells below

Run **exactly one** of the 4-bit / 16-bit / 32-bit cells. It sets `PRECISION` and `MODEL_KWARGS`,
which every section below uses (model loading, the on-disk folder `SAVED_MODEL_THINKINGCAP_QWEN36_27B_<precision>`, and the
Hub repo names `my-base-thinkingcap-qwen36-27B-<precision>-...`). To switch precision later, run a different cell,
then re-run the **Load the model** cell.

In [ ]:
# --- 4-bit (bitsandbytes NF4) ---
import torch
from transformers import BitsAndBytesConfig

PRECISION = "4bit"
MODEL_KWARGS = dict(
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    ),
    dtype=torch.bfloat16,
    device_map="cuda",
)
print("Selected precision:", PRECISION)

In [4]:
# --- 16-bit (half precision, bfloat16) ---
import torch

PRECISION = "16bit"
MODEL_KWARGS = dict(
    dtype=torch.bfloat16,
    device_map="auto",
)
print("Selected precision:", PRECISION)

Selected precision: 16bit


In [ ]:
# --- 32-bit (full precision, float32) ---
import torch

PRECISION = "32bit"
MODEL_KWARGS = dict(
    dtype=torch.float32,
    device_map="auto",
)
print("Selected precision:", PRECISION)

# Base model from HUB

**What this notebook does:** download the original `bottlecapai/ThinkingCap-Qwen3.6-27B` from the Hugging Face Hub at the precision you selected, verify it generates a coherent answer, then persist it — save a copy to local disk **and** push a copy to your own Hub repo.

```mermaid
flowchart LR
    ORIG["🤗 Original Hub<br/>ThinkingCap-Qwen3.6-27B"] -->|load| BM["base_model"]
    BM -->|test| OK["✅ generates"]
    BM -->|save| DISK["💾 disk<br/>SAVED_MODEL_&lt;prec&gt;"]
    BM -->|push| HUB["📦 Your Hub<br/>…-from-hub"]
    style ORIG fill:#2e7d32,color:#fff,stroke:#1b5e20
```

### Load the model

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Silence transformers' verbose warnings.
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

model_name = "bottlecapai/ThinkingCap-Qwen3.6-27B"

# Names derived from the precision you selected above -- used by every section below.
SAVED_DIR      = f"SAVED_MODEL_THINKINGCAP_QWEN36_27B_{PRECISION}"
REPO_FROM_HUB  = f"lukaskellerstein/my-base-thinkingcap-qwen36-27B-{PRECISION}-from-hub"
REPO_FROM_DISK = f"lukaskellerstein/my-base-thinkingcap-qwen36-27B-{PRECISION}-from-disk"

base_model = AutoModelForCausalLM.from_pretrained(model_name, **MODEL_KWARGS)
base_tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Loaded {model_name}  |  precision={PRECISION}  |  hub repo={REPO_FROM_HUB}")

model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

Loaded bottlecapai/ThinkingCap-Qwen3.6-27B  |  precision=16bit  |  hub repo=lukaskellerstein/my-base-thinkingcap-qwen36-27B-16bit-from-hub


Log model and tokenizer

In [12]:
def describe(model, tokenizer):
    """Print key facts about a loaded model + tokenizer. Reused in every section below."""
    print("--- Model ---")
    print("Type:", type(model))
    print("Architecture:", model)
    # get_text_config() returns the text sub-config for multimodal models and just
    # returns the config itself for plain text models (Qwen3.6 is text-only), so this
    # line works regardless of model family.
    print("Vocab size:", model.config.get_text_config().vocab_size)
    print("Input embeddings:", model.get_input_embeddings())
    print("Output embeddings:", model.get_output_embeddings())
    print("--- Tokenizer ---")
    print("Type:", type(tokenizer))
    print("Special tokens:", tokenizer.special_tokens_map)
    print("All tokens count:", len(tokenizer))
    print("Padding side:", tokenizer.padding_side)


describe(base_model, base_tokenizer)

--- Model ---
Type: <class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5ForCausalLM'>
Architecture: Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 5120)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (act): SiLUActivation()
          (conv1d): Conv1d(10240, 10240, kernel_size=(4,), stride=(1,), padding=(3,), groups=10240, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=6144, out_features=5120, bias=False)
          (in_proj_qkv): Linear(in_features=5120, out_features=10240, bias=False)
          (in_proj_z): Linear(in_features=5120, out_features=6144, bias=False)
          (in_proj_b): Linear(in_features=5120, out_features=48, bias=False)
          (in_proj_a): Linear(in_features=5120, out_features=48, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=5120, out_features=17408, bias=F

### Test the model

#### via Pipeline

In [13]:
from transformers import pipeline

PROMPT = "What is AutoGen in abstract?"


def test_pipeline(model, tokenizer, prompt=PROMPT):
    """Test a model with a text-generation pipeline. Reused in every section below."""
    my_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

    # ThinkingCap-Qwen3.6-27B is instruction-tuned, so we pass a chat (list of messages)
    # instead of a raw string. The pipeline then auto-applies the tokenizer's chat
    # template, giving a coherent answer (a raw string would ignore the template and ramble).
    # It's a reasoning model, so it may emit a <think>...</think> block before the answer --
    # give it plenty of tokens.
    messages = [{"role": "user", "content": prompt}]
    result = my_pipeline(messages, max_new_tokens=1024, return_full_text=False)
    print(result[0]["generated_text"])


test_pipeline(base_model, base_tokenizer)

Here's a thinking process:

1.  **Understand User Query**: The user asks "What is AutoGen in abstract?" This means they want a high-level, conceptual understanding of AutoGen, not a detailed technical breakdown or code examples.

2.  **Identify Core Subject**: AutoGen is a framework/library developed by Microsoft Research for building applications with multiple AI agents that can collaborate, communicate, and solve problems together.

3.  **Key Abstract Concepts**:
   - Multi-agent systems
   - AI agents that can converse/collaborate
   - Framework for orchestrating agent interactions
   - Enables complex task decomposition and execution
   - Human-in-the-loop capability
   - Built on LLMs but abstracts away low-level details

4.  **Structure the Abstract Explanation**:
   - Definition/High-level summary
   - Core idea/paradigm
   - Key characteristics (abstracted)
   - Purpose/Value proposition
   - Keep it concise and conceptual

5.  **Draft (Mental Refinement)**:
   AutoGen is an op

#### via Model and Tokenizer

In [14]:
def test_generate(model, tokenizer, prompt=PROMPT):
    """Test a model by manually applying the chat template + generate. Reused below."""
    # Instruction-tuned model -> build inputs from a chat via the tokenizer's chat template.
    # add_generation_prompt=True appends the assistant marker so the model actually
    # answers instead of continuing the user's turn.
    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
    ).to(model.device)

    # Generate, then decode only the answer. Reasoning model -> allow a long output.
    result = model.generate(**model_inputs, max_new_tokens=1024)
    print(tokenizer.decode(result[0], skip_special_tokens=True))


test_generate(base_model, base_tokenizer)

user
What is AutoGen in abstract?
assistant
<think>
Here's a thinking process:

1.  **Understand User Query**: The user asks "What is AutoGen in abstract?" This means they want a high-level, conceptual overview of AutoGen, focusing on its core ideas, purpose, and architecture rather than implementation details or specific code.

2.  **Identify Key Subject**: AutoGen is a framework developed by Microsoft Research. It's designed for building AI applications using multiple conversational agents that can collaborate, converse, and solve tasks together.

3.  **Extract Core Concepts (Abstract Level)**:
   - **Multi-Agent Collaboration**: Core idea is that multiple AI agents work together, each with specialized roles/capabilities.
   - **Conversational Interface**: Agents communicate via natural language/messages, mimicking human teamwork.
   - **Human-in-the-Loop**: Supports human participation as an agent or overseer.
   - **Task Automation & Complex Problem Solving**: Designed for complex 

#### via Tool call (single turn)

In [15]:
# Single-turn tool calling: pass `tools` to the chat template. An instruction-tuned
# model should answer with a tool call (function name + JSON arguments) instead of prose.
STOCK_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current price of a stock.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The ticker symbol for the stock, e.g. GOOG",
                    }
                },
                "required": ["ticker"],
            },
        },
    }
]


def test_tool_call(model, tokenizer, prompt, tools, max_new_tokens=1024):
    """Single-turn tool calling: pass `tools`; the model should emit a tool call. Reused below."""
    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)

    # Greedy decoding -> deterministic tool call. Keep special tokens so the
    # tool-call markers stay visible.
    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = result[0][model_inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(gen, skip_special_tokens=False))


test_tool_call(base_model, base_tokenizer, "What is the current price of MSFT?", STOCK_TOOLS)

Thinking Process:
1.  Identify the user's request: Get the current price of MSFT stock.
2.  Identify the required tool: `get_stock_price`.
3.  Identify the required parameter: `ticker` = "MSFT".
4.  Call the tool.
5.  Return the result.✅
</think>

<tool_call>
<function=get_stock_price>
<parameter=ticker>
MSFT
</parameter>
</function>
</tool_call><|im_end|>


#### via Conversation with tool calls

In [17]:
def test_tool_conversation(model, tokenizer, max_new_tokens=1024):
    """Multi-turn conversation that already contains an assistant tool call + a tool
    response, then a fresh user turn. Confirms the chat template round-trips the
    `tool_calls` / `tool` roles, then answers the latest turn. Reused below.

    NOTE (model-specific): Qwen3.5's chat template iterates the tool-call
    `arguments` with `.items()`, so `arguments` must be a **dict** here (Gemma's
    template accepted a JSON *string*). Passing a string raises
    "TypeError: Can only get item pairs from a mapping."."""
    conversation = [
        {"role": "user", "content": "What's the weather like in Paris?"},
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {
                    "id": "call12345",
                    "type": "function",
                    "function": {
                        "name": "get_current_weather",
                        "arguments": {"location": "Paris, France", "format": "celsius"},
                    },
                }
            ],
        },
        {"role": "tool", "content": '{"content": 22}', "tool_call_id": "call12345"},
        {
            "role": "assistant",
            "content": "The current temperature in Paris, France is 22 degrees Celsius.",
        },
        {"role": "user", "content": "What weather will be in San Francisco next week?"},
    ]

    weather_tools = [
        {
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "description": "Get the current weather",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA",
                        },
                        "format": {
                            "type": "string",
                            "enum": ["celsius", "fahrenheit"],
                            "description": "The temperature unit to use. Infer this from the users location.",
                        },
                    },
                    "required": ["location", "format"],
                },
            },
        }
    ]

    model_inputs = tokenizer.apply_chat_template(
        conversation, tools=weather_tools, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    print("Tokenized conversation input_ids:")
    print(model_inputs["input_ids"])

    # Answer the latest user turn (should be another get_current_weather tool call for SF).
    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = result[0][model_inputs["input_ids"].shape[1]:]
    print("\nModel response:")
    print(tokenizer.decode(gen, skip_special_tokens=False))


test_tool_conversation(base_model, base_tokenizer)

Tokenized conversation input_ids:
tensor([[248045,   8678,    198,      2,  13455,    271,   2523,    599,   2528,
            310,    279,   2614,   5568,     25,    271,     27,  15449,     29,
            198,   4754,   1267,    763,    328,   1628,    487,    328,   1628,
            763,   5046,    591,    763,    328,    447,  10757,  67017,    487,
            328,   4532,    763,    328,   1882,    279,   1428,   8831,    487,
            328,  13390,    763,   5046,   1267,    763,    328,   1640,    487,
            328,  12811,    763,   5046,   2447,    763,   5046,   1267,    763,
            328,    889,    487,    328,   4532,    763,    328,    760,   3177,
            321,   1528,     11,    378,   1257,     13,   5655,  12510,     11,
           8910,  13933,    328,   2171,    763,   5046,   1267,    763,    328,
            889,    487,    328,   8746,    763,   4241,     66,  38915,    487,
            328,     69,  46311,   7664,    328,   4532,    763,    328,   


Model response:
Here's a thinking process:

1.  **Analyze User Input:** The user is asking for the weather forecast for San Francisco for "next week".
2.  **Check Available Tools:** I have access to `get_current_weather`, which only provides *current* weather, not forecasts for future dates.
3.  **Identify Limitation:** The tool cannot fulfill the request for a future forecast ("next week").
4.  **Formulate Response:** I need to politely inform the user that I can only provide current weather information, not future forecasts, and offer to get the current weather for San Francisco if they'd like. I should stick to the facts and be helpful within my constraints.
5.  **Draft Response:** I don't have access to future weather forecasts, but I can check the current weather in San Francisco for you if you'd like! Just let me know. (Self-Correction/Refinement: Keep it concise and direct.) "I can only provide current weather information, not forecasts for next week. Would you like me to check

### Save the model (to disk)

In [19]:
base_model.save_pretrained(SAVED_DIR)

base_tokenizer.save_pretrained(SAVED_DIR)

Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

('SAVED_MODEL_THINKINGCAP_QWEN36_27B_16bit/tokenizer_config.json',
 'SAVED_MODEL_THINKINGCAP_QWEN36_27B_16bit/chat_template.jinja',
 'SAVED_MODEL_THINKINGCAP_QWEN36_27B_16bit/tokenizer.json')

### Push the model (to HUB)

In [ ]:
# push_to_hub() -> save_pretrained(save_original_format=True) can raise NotImplementedError
# in transformers 5.x, so save with save_original_format=False and upload the folder instead.
from huggingface_hub import upload_folder, create_repo

tmp = f"PUSH_TMP_{PRECISION}_from_hub"
create_repo(REPO_FROM_HUB, token=API_TOKEN, exist_ok=True)
base_model.save_pretrained(tmp, save_original_format=False)
base_tokenizer.save_pretrained(tmp)
upload_folder(repo_id=REPO_FROM_HUB, folder_path=tmp, token=API_TOKEN)